<div style="border-left: 5px solid #b7791f; background-color: #fff8e1; padding: 0.8em 1em; margin: 1em 0; border-radius: 4px;">
  <strong>Warning: AI-assisted materials</strong><br><br>
  These materials were developed with assistance from AI tools. All content has been reviewed and edited by the instructor, who takes final responsibility for its accuracy, clarity, and appropriateness for the course. Students should treat these materials as instructor-reviewed course content while applying the same critical judgment they would use with any technical material. Please report any suspected errors or unclear explanations to ghunt@wm.edu.
</div>

# Model Selection

So far we have essentially been talking aobut evaluating a **fixed model form** $A$. Up to this point, the story has been:

- we train a model $\hat f = A(D_{train})$  
- we evaluate it using some metric  
  $$
  \hat{\text{Perf}}(D_{eval}, \hat f)
  $$
- we interpret this as an estimate of performance on new data  

Either we tried to estimate the performance of $\hat{f}$, or more likely, we estimating the performance of $A$. 

However, in practice, we are almost never given a single model. Instead, we have:

- many possible models  
- many choices of differnent knobs, parameters, arguments, "hyperparameters"

For example:

- different polynomial degrees
- different model classes altogether  
- different values of regularization (later)

So the real question becomes: Which model should we use?

We are **using performance to make decisions**. This comes with some caveats we must pay attention to. Most prominently: **Once we use performance to make decisions, it is no longer just an unbiased measurement.** The evaluation metric is now guiding model selection and influencing which model we end up with. So, in essence, it becomes part of the **training process itself**.

## From evaluation to selection

- Suppose we have a family of algorithms:
  $$
  \{ A_\lambda : \lambda \in \Lambda \}
  $$

Instead of a single algo, we now have a **collection of candidates** indexed by a parameter $\lambda$. (Here, $\lambda$ is just indexing all the various choices we could make). For example, $\lambda$ could represent:

- polynomial degree  
- number of features  
- or even entirely different model classes
- other things like regularization, tree depth, neighbors, we'll discuss later for particular algorithms.

Our goal is to choose $\lambda$ that gives best performance.

More concretely, we want to find the value of $\lambda$ that leads to a model with the best generalization performance  (i.e., performs best on new, unseen data). Honestly, the difference between selecting the meta parameter $\lambda$ and tuning, e.g., weights $w$ within the algorithm is less imporant. They are both part of the **model building process**. 

In some sense, you are now defining a **meta-algorithm**. Previously, we had:

- a learning algorithm $A$  
- which takes data and outputs a model  

Now we are building a higher-level procedure that. For example:

1. For each $\lambda$:
   - train:
     $$
     \hat f_\lambda = A_\lambda(D_{train})
     $$
   - evaluate:
     $$
     \hat{\text{Perf}}(D_{eval}, \hat f_\lambda)
     $$

So for each candidate $\lambda$, we fit a model and then measure its performance. 

2. Choose:
   $$
   \hat \lambda = \arg\min_\lambda \hat{\text{Perf}}(D_{eval}, \hat f_\lambda)
   $$

We pick the value of $\lambda$ that performs best on the evaluation set.

3. Output:
   $$
   \hat f = A_{\hat \lambda}(D_{all})
   $$

This is the final model we will use.

We are now optimizing over, not just model parameters, but over the **space of models / hyperparameters**

In a larger sense, however, we have just turned evaluation into part of the learning algorithm.

The final model $\hat f$ depends on:

- the training data  
- the evaluation data  
- the selection procedure  

If we want, it can be helpful to package the entire process into a single **model building procedure**:
$$
\hat f = \mathcal{P}(D)
$$

where $\mathcal{P}$ includes training, evaluation, and model selection.

The pseudo-code for $\mathcal{P}$ is essentially works on a 

- dataset $D$

with choices of

- model family $\{A_\lambda : \lambda \in \Lambda\}$  
- evaluation metric $\hat{\text{Perf}}(\cdot, \cdot)$

It then (for example):

1. **Splits the data:**
   $$
   D = (D_{train}, D_{eval})
   $$

2. **For each $\lambda \in \Lambda$:**
   - train:
     $$
     \hat f_\lambda = A_\lambda(D_{train})
     $$
   - evaluate:
     $$
     e_\lambda = \hat{\text{Perf}}(D_{eval}, \hat f_\lambda)
     $$

3. **Select the best parameters:**
   $$
   \hat\lambda = \arg\min_\lambda e_\lambda
   $$

4. **Retrain the final model:**
   $$
   \hat f = A_{\hat\lambda}(D)
   $$

and it outputs the final model $\hat f$.

The “learning algorithm” is no longer just $A$, but the entire pipeline $\mathcal{P}$ **including $D_{eval}$.**

## Can we trust $D_{eval}$ now?

Once $\hat f$ depends on the evaluation data (through $\hat \lambda$), we can no longer treat that data as independent for evaluation. Previously, we required:
$$
D_{eval} \perp \hat f
$$
so that evaluation behaved like performance on new data.

But now, recall:
$$
\hat \lambda = \arg\min_\lambda \hat{\text{Perf}}(D_{eval}, \hat f_\lambda)
$$

and:
$$
\hat f = A_{\hat \lambda}(...)
$$

So the evaluation dataset is no longer just being *observed*, it is being **used to make decisions** via $\hat\lambda$, and $\hat\lambda$ depended (however strongly) on $D_{eval}$. 

We run into all the same problems we just discussed with evaluation. Its potentially true that $\hat{\text{Perf}}(D_{eval}, \hat f_{\hat\lambda})$ is an overly optmistic estimate of performance. Its "seen" the evaluation data. How big of a problem is it? It depends on "how independent" $D_{eval}$ is from the final model. This comes back to our discussion on model capacity and flexibility:

- How flexible is the model family $\Lambda$?  
- How many choices did we try?  
- How aggressively did we optimize over $\lambda$?  

If:

- the space $\Lambda$ is small  
- the models are not very flexible
- we didn't try too hard to optimize performance over $\lambda$

then the dependence on $D_{eval}$ may be weak, and the bias may be small.

But if:

- $\Lambda$ is large  
- the models are highly flexible  
- we search aggressively over many options  

then we are effectively overfitting to the evaluation set.


## Three-way split train/validation/test

To resolve this issue, we introduce a third dataset:

- $D_{train}$ → fit models  
- $D_{validation}$ → select $\lambda$  
- $D_{test}$ → final evaluation  

Why? The validation set is now part of the training procedure (via selection).  Once we use $D_{val}$ to choose $\hat\lambda$, the final model $\hat f$ depends on it. So we are back in the same situation as before:

$$
D_{val} \not\perp \hat f
$$

which means $D_{val}$ is no longer valid for unbiased evaluation.

The solution is actually relatively simple: **Use a completely separate dataset that is never used during training or selection.** That dataset is $D_{test}$. This will ensure that
$$
D_{test} \perp \hat f
$$

so that evaluation on $D_{test}$ behaves like performance on new data.

The full procedure now looks like this:

1. **Train models on $D_{train}$**

   For each $\lambda$:
   $$
   \hat f_\lambda = A_\lambda(D_{train})
   $$

2. **Select $\hat \lambda$ using $D_{val}$**

   $$
   \hat\lambda = \arg\min_\lambda \hat{\text{Perf}}(D_{val}, \hat f_\lambda)
   $$

3. **Retrain the final model**

   $$
   \hat f = A_{\hat\lambda}(D_{train} \cup D_{val})
   $$

   We now use all available data (except the test set) to fit the final model.

4. **Evaluate once on $D_{test}$**

   $$
   \hat{\text{Perf}}(D_{test}, \hat f)
   $$

The important idea is that the test set is used **once**, at the very end. It is never used to:

- choose $\lambda$  
- compare models  
- tune anything  

Splitting into $D_{train}$, $D_{val}$, and $D_{test}$ can feel like a bit of a hack, and in some sense, it is.

You can equivalently think of it as first split into train/test, and then, within the training process, split the training data again into train/validation.

Concretely:

1. Split:
   $$
   D = (D_{train}^{(full)}, D_{test})
   $$

2. During model selection, further split:
   $$
   D_{train}^{(full)} = (D_{train}, D_{val})
   $$

So the “three-way split” is really just a way of organizing the requirement that evaluation data must remain independent of the model whenever we're doing evaluation.


The key issue in model selection is not the evaluation metric itself, but how it interacts with the selection step. For any fixed $\lambda$, the evaluation is a valid estimate of performance, because the data $D_{val}$ is independent of $\hat f_\lambda$. So taken one at a time, each model is evaluated correctly. However, when we optimize over $\lambda$ we introduce a **coupling** between the final model and the evaluation data. The choice of $\hat\lambda$ is not arbitrary, it depends directly on how each model performs on $D_{val}$. So the final model is no longer independent of $D_{val}$. So individually the estiamtes are ok, but if our building procedure selects based on $D_{val}$ then evaluation on that set is not longer unbiased. 

## Nested cross-validation

The train/validation/test split solves the problem conceptually, but it can be inefficient when data is limited. One way around this is to use what is often called **nested cross-validation**. (This is really just a dressed-up version of cross-validation we already know.)

Previously, we described a simple approach:

- split data into train / validation / test  
- use validation for model selection  
- use test for final evaluation  

We also noted that this can be viewed as: first doing a train/test split, and then splitting the training data again into train/validation.

Let's call:

- the first split → the **outer split**  
- the second split → the **inner split**  

Nested cross-validation is essentially a refinement of this idea. Instead of performing the outer and inner splits just once, we perform them multiple times using cross-validation.

We can package ordinary cross-validation into a procedure:
$$
\texttt{cross\_validate}(D, A)
$$

that takes a dataset $D$ and algorithm $A$, and returns an estimated performance for a fixed learning algorithm.

For example, in $k$-fold cross-validation:

- Partition the dataset $D$ into $k$ disjoint subsets (folds): $D_1, \dots, D_k$
- For each $j = 1, \dots, k$:
  - define $D_{eval}^{(j)} = D_j$
  - define $D_{train}^{(j)} = D \setminus D_j$
  - train:
    $$
    \hat f^{(j)} = A(D_{train}^{(j)})
    $$
  - compute:
    $$
    \hat{\text{Perf}}(D_{eval}^{(j)}, \hat f^{(j)})
    $$

- Return a summary, e.g. the average:
$$
\texttt{cross\_validate}(D, A)
=
\frac{1}{k} \sum_{j=1}^k \hat{\text{Perf}}(D_{eval}^{(j)}, \hat f^{(j)})
$$

### Stepping stone

Now, given our overall data $D$, suppose we do this:

1. **Split the data into outer train-val and outer test**
   $$
   D = (D_{train\text{-}val}, D_{test})
   $$

2. **For each $\lambda$, estimate performance on the outer training portion**

   For each $\lambda \in \Lambda$:
   $$
   e_\lambda = \texttt{cross\_validate}(D_{train\text{-}val}, A_\lambda)
   $$

3. **Select the best $\lambda$**
   $$
   \hat\lambda = \arg\min_\lambda e_\lambda
   $$

4. **Retrain the final model on all of $D_{train\text{-}val}$**
   $$
   \hat f = A_{\hat\lambda}(D_{train\text{-}val})
   $$

5. **Evaluate once on the outer test set**
   $$
   \hat{\text{Perf}}(D_{test}, \hat f)
   $$

This is already the basic nested idea:

- use cross-validation inside $D_{train\text{-}val}$ to choose $\lambda$
- then evaluate the resulting model on a separate outer test set

### Next complication

We do not have to do just one outer train-val / test split. We can do this multiple times as well.

So let us package the above model-selection-and-retraining steps into a procedure
$$
\texttt{inner\_cv\_fit}(D_{train\text{-}val})
$$
which returns the fitted model
$$
\hat f = A_{\hat\lambda}(D_{train\text{-}val})
$$
where $\hat\lambda$ was chosen by cross-validation inside $D_{train\text{-}val}$.

Now we can perform cross-validation again at the outer level:

- Partition the overall dataset $D$ into $k$ disjoint subsets (folds): $D_1, \dots, D_k$
- For each $j = 1, \dots, k$:
  - define $D_{test}^{(j)} = D_j$
  - define $D_{train\text{-}val}^{(j)} = D \setminus D_j$
  - fit:
    $$
    \hat f^{(j)} = \texttt{inner\_cv\_fit}(D_{train\text{-}val}^{(j)})
    $$
  - compute:
    $$
    \hat{\text{Perf}}(D_{test}^{(j)}, \hat f^{(j)})
    $$

Then at the end, return a summary, e.g. the average:
$$
\frac{1}{k} \sum_{j=1}^k \hat{\text{Perf}}(D_{test}^{(j)}, \hat f^{(j)})
$$

The key point is:

> The inner loop is used for **model selection**, while the outer loop is used for **final performance estimation**.

This preserves the separation between:

- data used to choose the model  
- data used to evaluate the final procedure

**What are we estimating?** We're now estimating the performance of the `inner_cv_fit` procedure.

## Nested CV Pseudo-code

In [ ]:
def cross_validate(D, A, k):
    """
    Input:
        D : dataset
        A : fixed learning algorithm
        k : number of folds
    Output:
        estimated performance of A on D
    """
    split D into k folds: D_1, ..., D_k
    perf_values = []

    for j in {1, ..., k}:
        D_eval = D_j
        D_train = D \ D_j

        f_hat = A(D_train)
        perf_j = Perf_hat(D_eval, f_hat)

        append perf_j to perf_values

    return average(perf_values)


def inner_cv(D_train_val, {A_lambda : lambda in Lambda}, k_inner):
    """
    Input:
        D_train_val : dataset used for model selection
        {A_lambda}  : family of learning algorithms indexed by lambda
        k_inner     : number of inner folds
    Output:
        final fitted model after selecting lambda
    """
    perf_by_lambda = {}

    for lambda in Lambda:
        e_lambda = cross_validate(D_train_val, A_lambda, k_inner)
        perf_by_lambda[lambda] = e_lambda

    lambda_hat = argmin_lambda perf_by_lambda[lambda]

    # retrain on all of D_train_val using the selected lambda
    f_hat = A_lambda_hat(D_train_val)

    return f_hat


def nested_cross_validate(D, {A_lambda : lambda in Lambda}, k_outer, k_inner):
    """
    Input:
        D : full dataset
        {A_lambda} : model family
        k_outer : number of outer folds
        k_inner : number of inner folds
    Output:
        estimated performance of the full model-selection procedure
    """
    split D into k_outer folds: D_1, ..., D_k_outer
    outer_perf_values = []

    for j in {1, ..., k_outer}:
        D_test = D_j
        D_train_val = D \ D_j

        # perform model selection inside the training portion
        f_hat_j = inner_cv(D_train_val, {A_lambda : lambda in Lambda}, k_inner)

        # evaluate on the held-out outer test fold
        perf_j = Perf_hat(D_test, f_hat_j)

        append perf_j to outer_perf_values

    return average(outer_perf_values)

## Examples in code

### Simple train/val/test

First, lets show a simulation of what happens when we do a simple train/val/test split

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [ ]:
rng = np.random.default_rng()

In [ ]:
def true_function(x):
    return (
        0.6 * np.sin(2 * np.pi * x)
        + 0.45 * np.sin(8 * np.pi * x)
        + 0.3 * np.sin(14 * np.pi * x)
        + 1.0 * np.exp(-120 * (x - 0.2) ** 2)
        - 0.8 * np.exp(-100 * (x + 0.35) ** 2)
    )

def make_data(n, rng, noise_std=1.5):
    x = rng.uniform(-1.0, 1.0, size=n)
    y_true = true_function(x)
    y = y_true + rng.normal(0.0, noise_std, size=n)
    return x, y, y_true


def fit_poly_and_compute_mse(
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_eval: np.ndarray,
    y_eval: np.ndarray,
    degree: int,
) -> tuple[object, float, float]:
    model = make_pipeline(
        PolynomialFeatures(degree=degree, include_bias=True),
        LinearRegression(fit_intercept=False),
    )

    x_train_2d = x_train.reshape(-1, 1)
    x_eval_2d = x_eval.reshape(-1, 1)

    model.fit(x_train_2d, y_train)

    yhat_train = model.predict(x_train_2d)
    yhat_eval = model.predict(x_eval_2d)

    train_mse = mean_squared_error(y_train, yhat_train)
    eval_mse = mean_squared_error(y_eval, yhat_eval)

    return model, train_mse, eval_mse

In [ ]:
# Three-way split
x_train, y_train, _ = make_data(n=250, rng=rng)
x_val, y_val, _ = make_data(n=100, rng=rng)
x_test, y_test, _ = make_data(n=100, rng=rng)

max_degree = 30
degrees = np.arange(max_degree + 1)

train_mse = np.zeros_like(degrees, dtype=float)
val_mse = np.zeros_like(degrees, dtype=float)
test_mse = np.zeros_like(degrees, dtype=float)

# Fit one model per degree using only the training set
for i, degree in enumerate(degrees):
    model, tr_mse, va_mse = fit_poly_and_compute_mse(
        x_train, y_train, x_val, y_val, degree=degree
    )

    # Compute test error too, but do not use it for model selection
    yhat_test = model.predict(x_test.reshape(-1, 1))
    te_mse = mean_squared_error(y_test, yhat_test)

    train_mse[i] = tr_mse
    val_mse[i] = va_mse
    test_mse[i] = te_mse

# Select degree using validation error only
best_degree = int(degrees[np.argmin(val_mse)])

In [ ]:
print(f"Best degree by validation error: {best_degree}")
print(f"Training MSE at best degree:   {train_mse[best_degree]:.4f}")
print(f"Validation MSE at best degree: {val_mse[best_degree]:.4f}")
print(f"Test MSE at best degree:       {test_mse[best_degree]:.4f}")

In [ ]:
# Plot all three curves
plt.figure(figsize=(9, 5.5))
plt.plot(degrees, np.log10(train_mse), marker="o", label="Training error")
plt.plot(degrees, np.log10(val_mse), marker="o", label="Validation error")
plt.plot(degrees, np.log10(test_mse), marker="o", label="Test error")

plt.axvline(best_degree, linestyle="--", linewidth=1.5, label=f"Chosen degree = {best_degree}")

plt.xlabel("Polynomial degree")
plt.ylabel("Mean squared error")
plt.title("Train / validation / test error vs model complexity")
plt.xticks(degrees)
plt.legend()
plt.tight_layout()
plt.show()

### MC simulations

Now let's simulate things multiple times and look at the median so we can try to get rid of some of the noise effects:

In [ ]:
def fit_model(x_train, y_train, degree):
    model = make_pipeline(
        PolynomialFeatures(degree=degree, include_bias=True),
        LinearRegression(fit_intercept=False),
    )
    model.fit(x_train.reshape(-1, 1), y_train)
    return model


def run_experiment(
    max_degree=30,
    n_train=50,
    n_val=50,
    n_test=50,
    n_gt=20000,
    noise_std=1.5,
    rng=None,
):
    if rng is None:
        rng = np.random.default_rng()

    x_train, y_train, y_train_true = make_data(n_train, rng, noise_std=noise_std)
    x_val, y_val, y_val_true = make_data(n_val, rng, noise_std=noise_std)
    x_test, y_test, y_test_true = make_data(n_test, rng, noise_std=noise_std)

    # Dense "ground truth" sample, noiseless
    x_gt = np.linspace(-1.0, 1.0, n_gt)
    y_gt = true_function(x_gt)

    degrees = np.arange(max_degree + 1)

    train_err = []
    val_err = []
    test_err = []
    gt_err = []

    for d in degrees:
        model = fit_model(x_train, y_train, d)

        yhat_train = model.predict(x_train.reshape(-1, 1))
        yhat_val = model.predict(x_val.reshape(-1, 1))
        yhat_test = model.predict(x_test.reshape(-1, 1))
        yhat_gt = model.predict(x_gt.reshape(-1, 1))

        train_err.append(np.log10(mean_squared_error(y_train, yhat_train)))
        val_err.append(np.log10(mean_squared_error(y_val, yhat_val)))
        test_err.append(np.log10(mean_squared_error(y_test, yhat_test)))

        # Error against the noiseless underlying function on a large sample
        gt_err.append(np.log10(mean_squared_error(y_gt, yhat_gt)+noise_std**2))

    return {
        "degrees": degrees,
        "train_err": np.array(train_err),
        "val_err": np.array(val_err),
        "test_err": np.array(test_err),
        "gt_err": np.array(gt_err),
        "x_train": x_train,
        "y_train": y_train,
        "x_gt": x_gt,
        "y_gt": y_gt,
    }


def summarize(curves):
    median = np.median(curves, axis=0)
    q25 = np.percentile(curves, 25, axis=0)
    q75 = np.percentile(curves, 75, axis=0)
    return median, q25, q75

In [ ]:
rng = np.random.default_rng()

n_runs = 100
max_degree = 30
n_train=250
n_val=100
n_test=100
all_train = []
all_val = []
all_test = []
all_gt = []

for _ in range(n_runs):
    out = run_experiment(max_degree=max_degree, n_train=n_train, n_test=n_test, n_val=n_val, rng=rng)
    deg = out["degrees"]
    all_train.append(out["train_err"])
    all_val.append(out["val_err"])
    all_test.append(out["test_err"])
    all_gt.append(out["gt_err"])

all_train = np.array(all_train)
all_val = np.array(all_val)
all_test = np.array(all_test)
all_gt = np.array(all_gt)

tr_med, tr_q25, tr_q75 = summarize(all_train)
va_med, va_q25, va_q75 = summarize(all_val)
te_med, te_q25, te_q75 = summarize(all_test)
gt_med, gt_q25, gt_q75 = summarize(all_gt)

plt.figure(figsize=(10, 6))

# Median curves
plt.plot(deg, tr_med, label="Train (median)")
plt.plot(deg, va_med, label="Validation (median)")
plt.plot(deg, te_med, label="Test (median)")
plt.plot(deg, gt_med, label="Ground truth, dense noiseless sample (median)", linewidth=2)

# IQR bands
plt.fill_between(deg, tr_q25, tr_q75, alpha=0.15)
plt.fill_between(deg, va_q25, va_q75, alpha=0.15)
plt.fill_between(deg, te_q25, te_q75, alpha=0.15)
plt.fill_between(deg, gt_q25, gt_q75, alpha=0.15)

plt.xlabel("Model complexity (degree)")
plt.ylabel("log10(MSE)  [lower is better]")
plt.title("Train / Validation / Test / Ground-Truth Error")
plt.legend()
plt.tight_layout()
plt.show()

### Error at chosen degree

The fact that *on average* the test and val curves aren't that different isn't too surprising. They *should* estimate the same thing. The difference will arise in when we report the validation error at the value which minimizes this validation error. Let's plot the error at this minimum value for the various curves.

In [ ]:
def make_truth_grid(n_truth=100000):
    x_truth = np.linspace(-1.0, 1.0, n_truth)
    y_truth = true_function(x_truth)
    return x_truth, y_truth

def run_once(max_degree, n_train, n_val, n_test, n_truth, rng, noise_std=1.5):
    x_train, y_train, y_train_true = make_data(n_train, rng, noise_std=noise_std)
    x_val, y_val, y_val_true = make_data(n_val, rng, noise_std=noise_std)
    x_test, y_test, y_test_true = make_data(n_test, rng, noise_std=noise_std)

    # Dense, noiseless ground-truth sample
    x_truth, y_truth = make_truth_grid(n_truth)

    degrees = np.arange(max_degree + 1)

    train_err = []
    val_err = []
    test_err = []
    truth_err = []

    for d in degrees:
        model = fit_model(x_train, y_train, d)

        yhat_train = model.predict(x_train.reshape(-1, 1))
        yhat_val = model.predict(x_val.reshape(-1, 1))
        yhat_test = model.predict(x_test.reshape(-1, 1))
        yhat_truth = model.predict(x_truth.reshape(-1, 1))

        train_err.append(mean_squared_error(y_train, yhat_train))
        val_err.append(mean_squared_error(y_val, yhat_val))
        test_err.append(mean_squared_error(y_test, yhat_test))
        truth_err.append(mean_squared_error(y_truth, yhat_truth) + noise_std**2)

    train_err = np.array(train_err)
    val_err = np.array(val_err)
    test_err = np.array(test_err)
    truth_err = np.array(truth_err)

    # Select degree using validation error
    d_hat = np.argmin(val_err)

    return {
        "train_selected": train_err[d_hat],
        "val_selected": val_err[d_hat],
        "test_selected": test_err[d_hat],
        "truth_selected": truth_err[d_hat],
        "d_hat": d_hat,
        "degrees": degrees,
        "train_curve": train_err,
        "val_curve": val_err,
        "test_curve": test_err,
        "truth_curve": truth_err,
    }

In [ ]:
n_runs = 100
max_degree = 50

n_train = 250
n_val = 100
n_test = 100
n_truth = 1000

train_selected = []
val_selected = []
test_selected = []
truth_selected = []
chosen_degrees = []

all_train_curves = []
all_val_curves = []
all_test_curves = []
all_truth_curves = []

rng = np.random.default_rng()

for _ in range(n_runs):
    out = run_once(
        max_degree=max_degree,
        n_train=n_train,
        n_val=n_val,
        n_test=n_test,
        n_truth=n_truth,
        rng=rng,
    )
    train_selected.append(out["train_selected"])
    val_selected.append(out["val_selected"])
    test_selected.append(out["test_selected"])
    truth_selected.append(out["truth_selected"])
    chosen_degrees.append(out["d_hat"])

    all_train_curves.append(out["train_curve"])
    all_val_curves.append(out["val_curve"])
    all_test_curves.append(out["test_curve"])
    all_truth_curves.append(out["truth_curve"])

train_selected = np.array(train_selected)
val_selected = np.array(val_selected)
test_selected = np.array(test_selected)
truth_selected = np.array(truth_selected)
chosen_degrees = np.array(chosen_degrees)

all_train_curves = np.array(all_train_curves)
all_val_curves = np.array(all_val_curves)
all_test_curves = np.array(all_test_curves)
all_truth_curves = np.array(all_truth_curves)

In [ ]:
deg = np.arange(max_degree + 1)

plt.figure(figsize=(8, 5))
bins = np.arange(max_degree + 2) - 0.5
plt.hist(chosen_degrees, bins=bins, edgecolor="black")
plt.xticks(np.arange(max_degree + 1))
plt.xlabel("Chosen degree")
plt.ylabel("Count")
plt.title("Validation-selected model complexity")
plt.tight_layout()
plt.show()


print(f"Average selected train MSE: {train_selected.mean():.4f}")
print(f"Average selected val MSE:   {val_selected.mean():.4f}")
print(f"Average selected test MSE:  {test_selected.mean():.4f}")
print(f"Average selected truth MSE: {truth_selected.mean():.4f}")
print(f"Average chosen degree:      {chosen_degrees.mean():.2f}")

In [ ]:
plt.figure(figsize=(7, 5))

plt.boxplot(
    [train_selected, val_selected, test_selected, truth_selected],
    labels=["Train", "Validation", "Test", "Truth"],
)

plt.ylabel("MSE (selected model)")
plt.title("Selected model error across runs")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))

plt.boxplot(
    [np.log10(train_selected), np.log10(val_selected), np.log10(test_selected), np.log10(truth_selected)],
    labels=["Train", "Validation", "Test", "Truth"],
)

plt.ylabel("MSE (selected model)")
plt.title("Selected model error across runs")
plt.tight_layout()
plt.show()

Another useful way to look at this is by looking atoverfitting at the model-selection level by introducing the idea of meta-capacity. 

We split data into train, validation, and test sets, fit polynomial models of varying degree on the training data, and then, for each maximum degree K, select the best model from {0, ..., K} using validation error. The key variable is K, which controls how many models we are allowed to search over, i.e., the flexibility of the selection procedure itself: something like a meta-capacity. 

As K increases, the procedure has more opportunities to find a model that looks good on the validation set, leading to increasingly optimistic validation error.

In [ ]:
def true_function(x):
    return (
        0.6 * np.sin(2 * np.pi * x)
        + 0.45 * np.sin(8 * np.pi * x)
        + 0.3 * np.sin(14 * np.pi * x)
        + 1.0 * np.exp(-120 * (x - 0.2) ** 2)
        - 0.8 * np.exp(-100 * (x + 0.35) ** 2)
    )

def make_data(n, rng, noise_std=1.0):
    x = rng.uniform(-1.0, 1.0, size=n)
    y_true = true_function(x)
    y = y_true + rng.normal(0.0, noise_std, size=n)
    return x, y, y_true


def fit_model(x_train, y_train, degree):
    model = make_pipeline(
        PolynomialFeatures(degree=degree, include_bias=True),
        LinearRegression(fit_intercept=False),
    )
    model.fit(x_train.reshape(-1, 1), y_train)
    return model


def summarize(curves):
    median = np.median(curves, axis=0)
    q25 = np.percentile(curves, 25, axis=0)
    q75 = np.percentile(curves, 75, axis=0)
    return median, q25, q75


def run_one_meta_experiment(
    max_degree=30,
    n_train=30,
    n_val=50,
    n_test=50,
    noise_std=1.0,
    rng=None,
):
    if rng is None:
        rng = np.random.default_rng()

    x_train, y_train, _ = make_data(n_train, rng, noise_std=noise_std)
    x_val, y_val, _ = make_data(n_val, rng, noise_std=noise_std)
    x_test, y_test, _ = make_data(n_test, rng, noise_std=noise_std)

    degrees = np.arange(max_degree + 1)

    # Error for each fixed degree
    val_curve = np.zeros(max_degree + 1)
    test_curve = np.zeros(max_degree + 1)

    fitted_models = []

    for d in degrees:
        model = fit_model(x_train, y_train, d)
        fitted_models.append(model)

        yhat_val = model.predict(x_val.reshape(-1, 1))
        yhat_test = model.predict(x_test.reshape(-1, 1))

        val_curve[d] = mean_squared_error(y_val, yhat_val)
        test_curve[d] = mean_squared_error(y_test, yhat_test)

    # Meta-procedure curves:
    # for each K, search over degrees {0, ..., K}, choose by validation,
    # then report selected validation and selected test error.
    selected_val_by_K = np.zeros(max_degree + 1)
    selected_test_by_K = np.zeros(max_degree + 1)
    selected_degree_by_K = np.zeros(max_degree + 1, dtype=int)

    for K in degrees:
        best_d = np.argmin(val_curve[: K + 1])
        selected_degree_by_K[K] = best_d
        selected_val_by_K[K] = val_curve[best_d]
        selected_test_by_K[K] = test_curve[best_d]

    return {
        "degrees": degrees,
        "fixed_val_curve": val_curve,
        "fixed_test_curve": test_curve,
        "selected_val_by_K": selected_val_by_K,
        "selected_test_by_K": selected_test_by_K,
        "selected_degree_by_K": selected_degree_by_K,
    }

In [ ]:
rng = np.random.default_rng()

n_runs = 200
max_degree = 50

all_selected_val = []
all_selected_test = []
all_selected_degree = []

# keep one run of the fixed-degree curves for illustration
example_fixed_val = None
example_fixed_test = None

for run in range(n_runs):
    out = run_one_meta_experiment(
        max_degree=max_degree,
        n_train=250,
        n_val=100,
        n_test=100,
        noise_std=1.5,
        rng=rng,
    )

    if run == 0:
        example_fixed_val = out["fixed_val_curve"]
        example_fixed_test = out["fixed_test_curve"]

    all_selected_val.append(out["selected_val_by_K"])
    all_selected_test.append(out["selected_test_by_K"])
    all_selected_degree.append(out["selected_degree_by_K"])

all_selected_val = np.array(all_selected_val)
all_selected_test = np.array(all_selected_test)
all_selected_degree = np.array(all_selected_degree)

K_grid = np.arange(max_degree + 1)

val_med, val_q25, val_q75 = summarize(all_selected_val)
test_med, test_q25, test_q75 = summarize(all_selected_test)
deg_med, deg_q25, deg_q75 = summarize(all_selected_degree)

In [ ]:
# which degree tends to get selected as K grows
plt.figure(figsize=(9, 5))
plt.plot(K_grid, deg_med, label="Median selected degree", linewidth=2)
plt.fill_between(K_grid, deg_q25, deg_q75, alpha=0.15)
plt.xlabel("Meta-capacity K")
plt.ylabel("Selected degree")
plt.title("Selected degree as the search space expands")
plt.legend()
plt.tight_layout()
plt.show()


# meta-overfitting plot
plt.figure(figsize=(9, 5.5))

plt.plot(K_grid, val_med, label="Selected validation error", linewidth=2)
plt.plot(K_grid, test_med, label="Selected test error", linewidth=2)

plt.fill_between(K_grid, val_q25, val_q75, alpha=0.15)
plt.fill_between(K_grid, test_q25, test_q75, alpha=0.15)

plt.xlabel("Meta-capacity K (search over degrees 0,...,K)")
plt.ylabel("MSE of selected model")
plt.title("Meta-overfitting from validation-based degree selection")
plt.legend()
plt.tight_layout()
plt.show()

# meta-overfitting plot diff
plt.figure(figsize=(9, 5.5))

plt.plot(K_grid, test_med-val_med, label="Selected error optimism", linewidth=2)

plt.xlabel("Meta-capacity K (search over degrees 0,...,K)")
plt.ylabel("Optimism selected model")
plt.title("Optimism from validation-based degree selection")
plt.legend()
plt.tight_layout()
plt.show()


## Review Questions

See: @sec-model-sel-questions.